# Ridge model builder (no cross validation)

Builds one Ridge regression per sales target using the **already-selected best parameters**
from the earlier grid search. No cross validation is performed here.

A single `StandardScaler` is fit on training features and shared across all target models. Each target has its own Ridge model.

Outputs are grouped under `ml_models/<selection-date>_<metric>/`; rerunning replaces only that run folder:
* `scaler.joblib` and `scaler_metadata.json` - the shared fitted transformation
* `<target>/ridge.joblib` - one fitted Ridge model per target
* `feature_map.json` - the feature order every model expects
* `model_metadata.csv` - per-target weights, intercepts, hyperparameters and metrics. Pair it with the shared scaler metadata to predict without sklearn.
  `y = intercept + sum(weight_f * (x_f - scaler_mean_f) / scaler_scale_f)`

Standardized-feature weights describe associations, not causal importance.

In [1]:
import json
import shutil
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Configuration
Edit paths here only. Model hyperparameters, split settings, and random state come from the parameter CSV.

In [2]:
def find_project_dir():
    required_files = (
        'car_sales_prediction_targets.csv',
        'car_sales_prediction_features.csv',
        'best_parameters.csv',
    )
    working_dir = Path.cwd().resolve()
    search_roots = (working_dir, *working_dir.parents)
    candidates = []
    for root in search_roots:
        candidates.extend((root, root / 'predict_car_sales_ML_modeling'))
        candidates.extend(root.glob('*/predict_car_sales_ML_modeling'))
    for candidate in candidates:
        if all((candidate / filename).is_file() for filename in required_files):
            return candidate
    raise FileNotFoundError(
        f'Could not find project inputs and best_parameters.csv from {working_dir}. '
        'Run rate_to_carsales_predictor.ipynb with SAVE_RESULTS = True first.'
    )


In [3]:

PROJECT_DIR = find_project_dir()
TARGETS_PATH = PROJECT_DIR / 'car_sales_prediction_targets.csv'
FEATURES_PATH = PROJECT_DIR / 'car_sales_prediction_features.csv'
PARAMETERS_PATH = PROJECT_DIR / 'best_parameters.csv'
MODELS_DIR = PROJECT_DIR / 'ml_models'

# Column-name prefixes used in the metadata CSV, so columns are grouped and self-describing.
WEIGHT_PREFIX = 'weight__'
VALID_SELECTION_METRICS = {'r2', 'neg_mean_absolute_error'}

## Load and validate data
Dates must be unique, parseable, and identical in both files. No imputation or lagging is done.

In [4]:
targets = pd.read_csv(TARGETS_PATH)
features = pd.read_csv(FEATURES_PATH)

def find_date_column(frame, name):
    """Return the single column that looks like a date and parses fully."""
    candidates = [c for c in frame.columns if 'observation_date' in c.lower() or 'time' in c.lower()]
    valid = [c for c in candidates if pd.to_datetime(frame[c], errors='coerce').notna().all()]
    if len(valid) != 1:
        raise ValueError(f'{name}: ambiguous or invalid date columns {candidates}')
    return valid[0]


def normalise_dates(frame, name):
    col = find_date_column(frame, name)
    out = frame.copy()
    out['observation_date'] = pd.to_datetime(out[col])
    if col != 'observation_date':
        raise ValueError("observation_date was the expected date column, but wan't found")
    if out['observation_date'].duplicated().any():
        raise ValueError(f'{name} has duplicate dates')
    return out


targets = normalise_dates(targets, 'targets')
features = normalise_dates(features, 'features')

mn_date = targets['observation_date'].min()
features = features.loc[features['observation_date']>=mn_date]


if set(targets['observation_date']) != set(features['observation_date']):
    raise ValueError('Target and feature dates differ')

# Columns shared by both files (other than date) are identifiers, not model inputs.
shared = sorted((set(targets.columns) & set(features.columns)) - {'observation_date'})
target_columns = [c for c in targets.columns if c not in {'observation_date', *shared}]
# Feature order is fixed here and is the order every saved model expects.
feature_columns = [c for c in features.columns if c not in {'observation_date', *shared}]

merged = targets[['observation_date', *target_columns]].merge(
    features[['observation_date', *feature_columns]], on='observation_date', how='inner', validate='one_to_one')
if len(merged) != len(targets):
    raise ValueError('Merge dropped rows')
if merged[target_columns + feature_columns].isna().any().any():
    raise ValueError('Missing values found; no imputation is performed')

X = merged[feature_columns]
print(f'{len(merged)} rows, {len(target_columns)} targets, {len(feature_columns)} features')

153 rows, 21 targets, 308 features


## Load best parameters
The parameter file includes the evaluation date and metric used by grid search, plus one row per target.

In [5]:
params = pd.read_csv(PARAMETERS_PATH)
required_metadata = {'target', 'model_selection_date', 'model_selection_metric'}
missing_metadata = required_metadata - set(params.columns)
if missing_metadata:
    raise ValueError(
        f'{PARAMETERS_PATH} is missing {sorted(missing_metadata)}. '
        'Rerun rate_to_carsales_predictor.ipynb with SAVE_RESULTS = True.'
    )
if params['model_selection_date'].isna().any() or params['model_selection_metric'].isna().any():
    raise ValueError('Selection date and metric must be present for every target')
selection_dates = params['model_selection_date'].astype(str).unique()
selection_metrics = params['model_selection_metric'].astype(str).unique()
if len(selection_dates) != 1 or len(selection_metrics) != 1:
    raise ValueError('Parameter rows must all use one selection date and metric')
SELECTION_DATE = datetime.strptime(selection_dates[0], '%Y-%m-%d').date().isoformat()
SELECTION_METRIC = selection_metrics[0]
if SELECTION_METRIC not in VALID_SELECTION_METRICS:
    raise ValueError(f'Unsupported model-selection metric: {SELECTION_METRIC}')
if 'grid_scoring' in params and not params['grid_scoring'].astype(str).eq(SELECTION_METRIC).all():
    raise ValueError('model_selection_metric does not match grid_scoring')
params = params.set_index('target')
missing = set(target_columns) - set(params.index)
if missing:
    raise ValueError(f'No saved parameters for targets: {sorted(missing)}')
if params.index.has_duplicates:
    raise ValueError('Parameter file contains duplicate target rows')


def clean(value):
    """Convert NaN -> None and numpy scalars -> plain Python types."""
    if pd.isna(value):
        return None
    return value.item() if hasattr(value, 'item') else value


def int_or_none(value):
    value = clean(value)
    return None if value is None else int(value)


# Split settings must be identical across targets so every target shares one train/test split.
split_cols = ['split_test_size', 'split_train_size', 'split_shuffle', 'split_random_state']
if (params.loc[target_columns, split_cols].nunique(dropna=False) > 1).any():
    raise ValueError('Split settings differ between targets')
first = params.loc[target_columns[0]]
scaler_cols = ['scaler_copy', 'scaler_with_mean', 'scaler_with_std']
if (params.loc[target_columns, scaler_cols].nunique(dropna=False) > 1).any():
    raise ValueError('Scaler settings differ between targets; a shared scaler cannot be used')
scaler_settings = params.loc[target_columns[0], scaler_cols]
SPLIT = {
    'test_size': float(first['split_test_size']),
    'train_size': float(first['split_train_size']),
    'shuffle': bool(first['split_shuffle']),
    'random_state': int_or_none(first['split_random_state']),
}
print('Split settings:', SPLIT)
print(f'Selection: {SELECTION_DATE} using {SELECTION_METRIC}')

Split settings: {'test_size': 0.35, 'train_size': 0.65, 'shuffle': True, 'random_state': 32}
Selection: 2026-11-18 using r2


## Train/test split
Split row positions once so all targets are trained and evaluated on the same rows.

In [6]:
train_idx, test_idx = train_test_split(np.arange(len(merged)), **SPLIT)
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
print(f'train={len(train_idx)} test={len(test_idx)}')

train=99 test=54


## Fit one shared scaler and Ridge models
The shared scaler learns mean/std from training rows only (no test leakage). One low-level Ridge model is fit per target on those transformed features.

In [7]:
MODEL_RUN_DIR = MODELS_DIR / f'{SELECTION_DATE}_{SELECTION_METRIC}'
if MODEL_RUN_DIR.exists():
    if MODEL_RUN_DIR.resolve().parent != MODELS_DIR.resolve():
        raise RuntimeError(f'Refusing to delete unexpected path {MODEL_RUN_DIR}')
    shutil.rmtree(MODEL_RUN_DIR)
MODEL_RUN_DIR.mkdir(parents=True, exist_ok=True)

scaler = StandardScaler(
    copy=bool(scaler_settings['scaler_copy']),
    with_mean=bool(scaler_settings['scaler_with_mean']),
    with_std=bool(scaler_settings['scaler_with_std']),
)
Xs_train = scaler.fit_transform(X_train)
Xs_test = scaler.transform(X_test)
joblib.dump(scaler, MODEL_RUN_DIR / 'scaler.joblib')
scaler_metadata = {
    'class': type(scaler).__name__,
    'parameters': scaler.get_params(),
    'mean': None if scaler.mean_ is None else scaler.mean_.tolist(),
    'scale': None if scaler.scale_ is None else scaler.scale_.tolist(),
}
(MODEL_RUN_DIR / 'scaler_metadata.json').write_text(json.dumps(scaler_metadata, indent=2))

rows = {}
for target in target_columns:
    p = params.loc[target]
    y_train, y_test = merged[target].iloc[train_idx], merged[target].iloc[test_idx]
    ridge = Ridge(
        alpha=float(p['ridge_alpha']),
        fit_intercept=bool(p['ridge_fit_intercept']),
        copy_X=bool(p['ridge_copy_X']),
        max_iter=int_or_none(p['ridge_max_iter']),
        tol=float(p['ridge_tol']),
        solver=p['ridge_solver'],
        positive=bool(p['ridge_positive']),
        random_state=int_or_none(p['ridge_random_state']),
    )
    ridge.fit(Xs_train, y_train)
    pred_train, pred_test = ridge.predict(Xs_train), ridge.predict(Xs_test)

    manual = ridge.intercept_ + Xs_test @ ridge.coef_
    if not np.allclose(manual, pred_test):
        raise AssertionError(f'Manual prediction does not match sklearn for {target}')

    target_dir = MODEL_RUN_DIR / target
    target_dir.mkdir()
    joblib.dump(ridge, target_dir / 'ridge.joblib')

    row = {'intercept': float(ridge.intercept_)}
    row.update({f'{WEIGHT_PREFIX}{f}': w for f, w in zip(feature_columns, ridge.coef_)})
    row.update({
        'ridge_alpha': ridge.alpha, 'ridge_solver': ridge.solver, 'ridge_fit_intercept': ridge.fit_intercept,
        'ridge_tol': ridge.tol, 'ridge_positive': ridge.positive,
        'split_test_size': SPLIT['test_size'], 'split_train_size': SPLIT['train_size'],
        'split_shuffle': SPLIT['shuffle'], 'split_random_state': SPLIT['random_state'],
        'model_selection_date': SELECTION_DATE, 'model_selection_metric': SELECTION_METRIC,
        'n_features': len(feature_columns), 'n_train': len(train_idx), 'n_test': len(test_idx),
        'train_r2': r2_score(y_train, pred_train), 'test_r2': r2_score(y_test, pred_test),
        'train_mae': mean_absolute_error(y_train, pred_train), 'test_mae': mean_absolute_error(y_test, pred_test),
        'target_mean_train': y_train.mean(), 'target_mean_test': y_test.mean(),
        'sklearn_version': sklearn.__version__,
    })
    rows[target] = row

## Save feature map and metadata CSV
`feature_map.json` lists the feature order (position 0, 1, 2, ...) all models expect; inputs must be
arranged in this order before calling any saved scaler or model.

In [8]:
feature_map = {
    'description': 'Feature name -> column position expected by the shared scaler.joblib and each target ridge.joblib.',
    'n_features': len(feature_columns),
    'features': {name: i for i, name in enumerate(feature_columns)},
}
(MODEL_RUN_DIR / 'feature_map.json').write_text(json.dumps(feature_map, indent=2))

metadata_df = pd.DataFrame.from_dict(rows, orient='index')
metadata_df.index.name = 'target'
metadata_path = MODEL_RUN_DIR / 'model_metadata.csv'
metadata_df.to_csv(metadata_path)
print(f'Saved {len(metadata_df)} targets x {metadata_df.shape[1]} columns to {metadata_path}')
print(f'Selection: {SELECTION_DATE} using {SELECTION_METRIC}')
print(f'Built {datetime.now():%Y-%m-%d %H:%M:%S}')

Saved 21 targets x 330 columns to /home/lelandmesford/Projects/dbt_projects/predict_car_sales_ML_modeling/ml_models/2026-11-18_r2/model_metadata.csv
Selection: 2026-11-18 using r2
Built 2026-11-18 13:31:22


## Review
Metrics per target, then a round-trip check that reloads the saved joblib files and a CSV-only prediction.

In [9]:
metadata_df[['ridge_alpha', 'train_r2', 'test_r2', 'train_mae', 'test_mae', 'target_mean_train', 'target_mean_test']]

,ridge_alpha,train_r2,test_r2,train_mae,test_mae,target_mean_train,target_mean_test
target,,,,,,,
commercial_van,3000.0,0.285301,0.290165,7.401231e+04,7.172591e+04,2.758394e+05,2.870529e+05
commercial_van_pct_of_total,3000.0,0.138900,0.089623,2.490113e-03,2.055867e-03,2.428283e-02,2.505556e-02
coupe_convertible,3000.0,0.180490,0.089104,7.468218e+03,8.299486e+03,1.011246e+04,1.269624e+04
coupe_convertible_pct_of_total,30.0,0.835275,0.487526,2.719326e-04,5.193820e-04,7.474747e-04,9.629630e-04
hatchback_liftback,3000.0,0.450658,0.567720,6.028878e+04,5.427361e+04,1.872121e+05,1.941284e+05
hatchback_liftback_pct_of_total,3000.0,0.383523,0.565222,3.835442e-03,3.330435e-03,1.524242e-02,1.562963e-02
minivan,3000.0,0.490231,0.467357,7.224311e+04,7.019744e+04,2.457160e+05,2.559409e+05
minivan_pct_of_total,3000.0,0.222406,0.227793,3.841597e-03,3.521191e-03,2.214141e-02,2.253704e-02
mixed_body_style,3000.0,0.507058,0.513981,3.667957e+05,3.722664e+05,1.181845e+06,1.283116e+06


In [10]:
# Round trip: load saved artifacts and the saved CSV, and confirm every route gives the same predictions.
saved_order = list(json.loads((MODEL_RUN_DIR / 'feature_map.json').read_text())['features'])
saved_meta = pd.read_csv(metadata_path, index_col='target')
X_check = X_test[saved_order]
saved_scaler = joblib.load(MODEL_RUN_DIR / 'scaler.joblib')
Xs_check = saved_scaler.transform(X_check)
scaler_info = json.loads((MODEL_RUN_DIR / 'scaler_metadata.json').read_text())
mean = np.zeros(len(saved_order)) if scaler_info['mean'] is None else np.asarray(scaler_info['mean'])
scale = np.ones(len(saved_order)) if scaler_info['scale'] is None else np.asarray(scaler_info['scale'])
for target in target_columns:
    ridge = joblib.load(MODEL_RUN_DIR / target / 'ridge.joblib')
    via_joblib = ridge.predict(Xs_check)

    r = saved_meta.loc[target]
    w = r[[f'{WEIGHT_PREFIX}{f}' for f in saved_order]].to_numpy(float)
    via_csv = r['intercept'] + ((X_check.to_numpy() - mean) / scale) @ w
    assert np.allclose(via_joblib, via_csv), target
print('Joblib and CSV-weight predictions agree for all targets.')

Joblib and CSV-weight predictions agree for all targets.
